# 1단계 GraphSAGE 실험: 집계 방식 · 레이어 수 · 학습률

1단계 레시피 유사도 모델(GTE 원본 입력, hidden 64)에서 **GraphSAGE 부분만 바꿔** 세 가지 실험을 합니다. 학습 절차, 손실(RMSE + 순서형), 5-폴드 교대, 조기 종료, 평가 지표(positive = 3점 이상)는 `training/stage1.py train`과 같습니다. 집계 방식을 `weighted_mean`, 레이어를 2로 두면 기존 모델과 같은 시드에서 **완전히 같은 결과**가 나옵니다.

**준비:** Colab 파일 탭에서 `data/stage1_rebuild_raw_gte.pt`를 `/content/data/`에 올립니다. GPU 런타임이면 GPU로, 아니면 CPU로 자동 학습합니다(`torch.cuda.is_available()`). CPU는 단일 스레드로 `training/stage1.py`와 비트 단위까지 같은 결과를 내고, GPU는 연산 순서가 달라 수치가 조금 다릅니다. 같은 장치 안에서는 같은 시드면 같은 결과이며, 장치가 다른 결과는 이어 하기에서 섞이지 않습니다. **한 실험 안의 조건 비교는 같은 장치에서** 하세요.

**실행:** [1] 공통 코드 → [2] 공통 설정을 먼저 실행한 뒤 실험 셀을 순서대로 실행합니다. 각 실험 셀 위쪽의 변수(`CONDITIONS_1`, `LAYERS`, `LRS` 등)를 바꾸고 다시 실행하면 조건을 바꿀 수 있습니다.

| 셀 | 내용 |
|---|---|
| [1] 공통 코드 | 실험용 모델(집계 방식·레이어 수 선택 가능), 학습 루프, 지표, 요약·그래프 함수 |
| [2] 공통 설정 | 시드 목록, 데이터 경로 확인 |
| 실험 1 | 집계 방식: 가중 평균(현재) / 일반 평균 / max pooling / GRU(셔플 미포함) / GRU(셔플 포함) |
| 실험 2 | GraphSAGE 레이어 수: 1 / 2 / 3 (집계 방식은 실험 1의 1위를 자동 사용) |
| 실험 3 | 학습률 (집계 방식·레이어 수는 실험 2의 1위를 자동 사용) |
| 결과 다운로드 | `results/experiments/` 압축 파일 다운로드 |

**결과 저장과 이어 하기:** 학습 1회마다 `results/experiments/<실험>/<조건>/seed<시드>/`에 `metrics.json`, `history.json`을 저장하고, 실험마다 `runs.csv`(시드별)와 `summary.csv`/`summary.json`(조건별 평균·표준편차)을 만듭니다. 같은 설정의 결과가 이미 있으면 다시 학습하지 않으므로, 조건을 추가하고 셀을 다시 실행하면 **새 조건만** 학습합니다. Colab 런타임이 끊기면 파일이 사라지니 마지막 셀로 내려받아 두세요.

**결과 읽는 법:** 조건마다 시드 5개로 학습해 평균 ± 표준편차를 봅니다. 검증 세트가 478쌍뿐이라 시드에 따른 차이가 작지 않습니다. 표에서 1위와의 검증 RMSE 차이가 1위의 표준편차보다 작은 조건에는 **≈**를 붙입니다(시드 노이즈 수준이라 우열을 가리기 어려움). 조건 선택은 **검증 RMSE 평균**으로 하고, 테스트 지표는 참고용입니다.

In [ ]:
# ======================================================================
# [1] 공통 코드: 모델·학습 루프·평가 지표 (training/stage1.py train과 동일 절차)
# ======================================================================
from __future__ import annotations

import copy
import csv
import json
import math
import os
import random
import statistics
from dataclasses import asdict, dataclass, replace
from pathlib import Path

# CUDA에서 결정론적 행렬곱(cuBLAS)에 필요: CUDA 초기화 전에 설정해야 함
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import torch
from torch import nn
from torch.nn import functional as F

# GPU가 있으면 GPU, 없으면 CPU로 학습
# (CPU는 단일 스레드로 training/stage1.py와 비트 단위까지 같은 결과, GPU는 연산 순서가 달라 수치가 조금 다름)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA = Path("data/stage1_rebuild_raw_gte.pt")
RESULTS = Path("results/experiments")
THRESHOLDS = (0.2, 0.4, 0.6, 0.8)  # 0~1 유사도에서 0/1/2/3/4점을 나누는 경계
AGGREGATORS = ("weighted_mean", "mean", "max", "gru")
GRU_ORDERS = ("fixed", "shuffle", "weight")


# ---------------- 데이터·손실·지표 (training/stage1.py와 동일) ----------------


def read_artifact(path: Path) -> dict:
    data = torch.load(path, map_location="cpu", weights_only=True)
    ids = data["recipe_ids"]
    features = data["recipe_features"]
    splits = data["splits"]
    if features.ndim != 2 or len(ids) != len(features):
        raise ValueError("unexpected recipe features")
    if not all(splits[part] for part in ("train", "val", "test")):
        raise ValueError("train, validation and test splits must be non-empty")
    if not torch.isfinite(features).all():
        raise ValueError("non-finite recipe features")
    return data


def context_edges(
    rows: list[tuple[int, int, int]],
) -> tuple[torch.Tensor, torch.Tensor]:
    positive = [(a, b, score / 4) for a, b, score in rows if score >= 1]
    if not positive:
        return torch.empty((2, 0), dtype=torch.long), torch.empty(0)
    forward = torch.tensor([(a, b) for a, b, _ in positive], dtype=torch.long).t()
    weights = torch.tensor([weight for _, _, weight in positive], dtype=torch.float32)
    return torch.cat((forward, forward.flip(0)), dim=1), weights.repeat(2)


def pair_tensors(
    rows: list[tuple[int, int, int]],
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    pairs = torch.tensor([(a, b) for a, b, _ in rows], dtype=torch.long).T.contiguous()
    classes = torch.tensor([score for _, _, score in rows], dtype=torch.long)
    labels = classes.float() / 4
    return pairs, labels, classes


def ordinal_loss(
    predictions: torch.Tensor,
    classes: torch.Tensor,
    temperature: float,
    last_boundary_weight: float = 1.0,
) -> torch.Tensor:
    thresholds = predictions.new_tensor(THRESHOLDS)
    targets = (classes[:, None] >= torch.arange(1, 5, device=classes.device)).float()
    logits = (predictions[:, None] - thresholds[None, :]) / temperature
    positive_weights = predictions.new_tensor((1, 1, 1, last_boundary_weight))
    return F.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=positive_weights
    )


def measure(predictions: torch.Tensor, classes: torch.Tensor) -> dict:
    # 0~4점 변환: floor(5 * 유사도), positive 기준: 3점 이상
    predicted = (predictions * 5).floor().long().clamp(0, 4)
    labels = classes.float() / 4
    matrix = [
        [int(((classes == actual) & (predicted == guess)).sum()) for guess in range(5)]
        for actual in range(5)
    ]
    positive_actual, positive_predicted = classes >= 3, predicted >= 3
    tp = int((positive_actual & positive_predicted).sum())
    fp = int((~positive_actual & positive_predicted).sum())
    fn = int((positive_actual & ~positive_predicted).sum())
    tn = int((~positive_actual & ~positive_predicted).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    specificity = tn / (tn + fp) if tn + fp else 0.0
    return {
        "n": len(classes),
        "rmse_0_1": (predictions - labels).square().mean().sqrt().item(),
        "mae_0_4": (predictions - labels).abs().mean().item() * 4,
        "exact_accuracy": (predicted == classes).float().mean().item(),
        "within_one_accuracy": ((predicted - classes).abs() <= 1).float().mean().item(),
        "positive_precision": precision,
        "positive_recall": recall,
        "positive_f1": (
            2 * precision * recall / (precision + recall) if precision + recall else 0.0
        ),
        "balanced_accuracy": (recall + specificity) / 2,
        "confusion_matrix": matrix,
        "predicted_counts": torch.bincount(predicted, minlength=5).tolist(),
        "actual_counts": torch.bincount(classes, minlength=5).tolist(),
        "per_class_recall": [
            matrix[k][k] / sum(matrix[k]) if sum(matrix[k]) else 0.0 for k in range(5)
        ],
        "min_prediction": predictions.min().item(),
        "max_prediction": predictions.max().item(),
    }


# ---------------- 모델 ----------------


class NeighborAggregator(nn.Module):
    # 노드별 이웃 벡터를 하나로 모음 [노드 수, hidden] (이웃이 없는 노드는 0 벡터)
    def __init__(self, kind: str, hidden: int, gru_order: str = "fixed"):
        super().__init__()
        if kind not in AGGREGATORS:
            raise ValueError(f"unknown aggregator: {kind}")
        if gru_order not in GRU_ORDERS:
            raise ValueError(f"unknown gru_order: {gru_order}")
        self.kind = kind
        self.gru_order = gru_order
        if kind == "max":
            self.pool = nn.Linear(hidden, hidden)
        elif kind == "gru":
            self.gru = nn.GRU(hidden + 1, hidden, batch_first=True)

    def forward(
        self,
        x: torch.Tensor,
        edges: torch.Tensor,
        weights: torch.Tensor,
        generator: torch.Generator | None = None,
    ) -> torch.Tensor:
        if self.kind == "weighted_mean":
            # training/stage1.py WeightedSAGELayer와 같은 연산 순서
            neighbors = torch.zeros_like(x)
            counts = x.new_zeros(len(x))
            if weights.numel():
                source, target = edges
                neighbors.index_add_(0, target, x[source] * weights[:, None])
                counts.index_add_(0, target, weights)
            return neighbors / counts.clamp_min(1e-8)[:, None]
        if not edges.numel():
            return torch.zeros_like(x)
        source, target = edges
        if self.kind == "mean":
            neighbors = torch.zeros_like(x).index_add(0, target, x[source])
            counts = x.new_zeros(len(x)).index_add(0, target, torch.ones_like(weights))
            return neighbors / counts.clamp_min(1)[:, None]
        if self.kind == "max":
            messages = F.relu(self.pool(x[source])) * weights[:, None]
            # 메시지가 모두 0 이상이므로 0에서 시작해도 이웃 최댓값과 같음
            return torch.zeros_like(x).scatter_reduce(
                0, target[:, None].expand_as(messages), messages, reduce="amax"
            )
        return self._gru(x, source, target, weights, generator)

    def _neighbor_order(
        self,
        target: torch.Tensor,
        weights: torch.Tensor,
        generator: torch.Generator | None,
    ) -> torch.Tensor:
        # 엣지를 대상 노드별로 모으고, 노드 안에서는 gru_order 순서로 놓는 순열
        if self.gru_order == "shuffle" and self.training:
            # 셔플 난수 생성기는 CPU에 두고 순열만 학습 장치로 옮김
            first = torch.randperm(len(target), generator=generator).to(target.device)
        elif self.gru_order == "weight":
            first = torch.argsort(weights, stable=True)
        else:  # fixed (shuffle 모델의 평가 포함): 엣지 목록 순서
            first = torch.arange(len(target), device=target.device)
        return first[torch.argsort(target[first], stable=True)]

    def _gru(self, x, source, target, weights, generator) -> torch.Tensor:
        order = self._neighbor_order(target, weights, generator)
        source, target, weights = source[order], target[order], weights[order]
        nodes, counts = torch.unique_consecutive(target, return_counts=True)
        starts = torch.cumsum(counts, 0) - counts
        rows = torch.repeat_interleave(torch.arange(len(nodes), device=x.device), counts)
        positions = torch.arange(len(target), device=x.device) - torch.repeat_interleave(
            starts, counts
        )
        sequences = x.new_zeros(len(nodes), int(counts.max()), x.shape[1] + 1)
        sequences[rows, positions] = torch.cat((x[source], weights[:, None]), dim=1)
        packed = nn.utils.rnn.pack_padded_sequence(
            sequences, counts.cpu(), batch_first=True, enforce_sorted=False
        )
        _, state = self.gru(packed)
        return torch.zeros_like(x).index_copy(0, nodes, state[-1])


class SAGELayer(nn.Module):
    # GraphSAGE 계층: 자기 벡터 변환 + 집계한 이웃 벡터 변환
    def __init__(self, hidden: int, aggregator: str, gru_order: str):
        super().__init__()
        self.self_linear = nn.Linear(hidden, hidden)
        self.neighbor_linear = nn.Linear(hidden, hidden, bias=False)
        self.aggregate = NeighborAggregator(aggregator, hidden, gru_order)

    def forward(self, x, edges, weights, generator=None) -> torch.Tensor:
        # 이웃 집계를 먼저 계산해야 기존 모델과 역전파 합산 순서가 같아 결과가 동일함
        neighbors = self.aggregate(x, edges, weights, generator)
        return self.self_linear(x) + self.neighbor_linear(neighbors)


class ExperimentPairGraphSAGE(nn.Module):
    """RecipePairGraphSAGE에서 집계 방식과 레이어 수만 바꿀 수 있게 한 실험용 모델

    레이어 구성은 기존과 같다: 입력 사영 -> [dropout -> SAGE -> LayerNorm -> GELU] x num_layers
    (2번째 레이어부터 residual 연결). num_layers=0이면 그래프를 쓰지 않는 기준 모델이다.
    """

    def __init__(
        self,
        input_dim: int,
        hidden: int,
        dropout: float,
        raw_cosine_stats: tuple[float, float] | None = None,
        num_layers: int = 2,
        aggregator: str = "weighted_mean",
        gru_order: str = "fixed",
    ):
        super().__init__()
        if num_layers < 0:
            raise ValueError("num_layers must be non-negative")
        self.dropout = dropout
        self.use_raw_cosine = raw_cosine_stats is not None
        if raw_cosine_stats is not None:
            self.register_buffer("raw_cosine_mean", torch.tensor(raw_cosine_stats[0]))
            self.register_buffer("raw_cosine_std", torch.tensor(raw_cosine_stats[1]))
        self.input = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.LayerNorm(hidden), nn.GELU()
        )
        # 기존 모델과 같은 순서로 파라미터를 만들어 같은 시드면 같은 초기값이 되게 함
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(SAGELayer(hidden, aggregator, gru_order))
            self.norms.append(nn.LayerNorm(hidden))
        self.pair_head = nn.Sequential(
            nn.Linear(hidden * 3 + int(self.use_raw_cosine), hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def encode(
        self,
        features: torch.Tensor,
        edges: torch.Tensor,
        weights: torch.Tensor,
        generator: torch.Generator | None = None,
    ) -> torch.Tensor:
        x = self.input(features)
        for index, (layer, norm) in enumerate(zip(self.layers, self.norms)):
            x = F.dropout(x, p=self.dropout, training=self.training)
            hidden = layer(x, edges, weights, generator)
            x = F.gelu(norm(hidden + x if index else hidden))
        return x

    def score(
        self,
        embeddings: torch.Tensor,
        pairs: torch.Tensor,
        raw_features: torch.Tensor | None = None,
    ) -> torch.Tensor:
        left, right = embeddings[pairs[0]], embeddings[pairs[1]]
        symmetric = torch.cat(
            (torch.abs(left - right), left * right, (left + right) / 2), dim=-1
        )
        if self.use_raw_cosine:
            raw_cosine = F.cosine_similarity(
                raw_features[pairs[0]], raw_features[pairs[1]]
            )
            standardized = (raw_cosine - self.raw_cosine_mean) / self.raw_cosine_std
            symmetric = torch.cat((symmetric, standardized[:, None]), dim=-1)
        return torch.sigmoid(self.pair_head(symmetric).squeeze(-1))


# ---------------- 학습 (training/stage1.py train과 같은 절차) ----------------


@dataclass(frozen=True)
class Config:
    # 기본값 = 현재 1단계 최적 설정 (GTE 원본 입력, hidden 64)
    aggregator: str = "weighted_mean"
    gru_order: str = "fixed"
    num_layers: int = 2
    lr: float = 0.003
    hidden: int = 64
    dropout: float = 0.15
    weight_decay: float = 0.001
    ordinal_weight: float = 0.2
    temperature: float = 0.1
    folds: int = 5
    epochs: int = 100
    patience: int = 10
    raw_cosine_feature: bool = True
    seed: int = 42
    # 학습 장치도 설정에 포함: CPU/GPU 결과는 수치가 달라 이어 하기 캐시에서 섞이지 않게 함
    device: str = DEVICE


def train_once(config: Config, data_path: Path = DATA, verbose: bool = False) -> dict:
    """설정 하나를 학습하고 metrics.json과 같은 형식의 결과(+history)를 반환"""
    if config.ordinal_weight < 0 or config.temperature <= 0 or config.folds < 2:
        raise ValueError("invalid training settings")
    device = torch.device(config.device)
    # 결정론적 실행: 같은 데이터·시드·장치면 항상 같은 결과
    if device.type == "cpu":
        torch.set_num_threads(1)
        torch.use_deterministic_algorithms(True)
    else:
        # 결정론적 구현이 없는 CUDA 연산은 오류 대신 경고 (해당 연산만 미세하게 달라질 수 있음)
        torch.use_deterministic_algorithms(True, warn_only=True)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    torch.manual_seed(config.seed)
    random.seed(config.seed)
    artifact = read_artifact(Path(data_path))
    features = artifact["recipe_features"]
    splits = artifact["splits"]
    raw_cosine_stats = None
    if config.raw_cosine_feature:
        training_pairs, _, _ = pair_tensors(splits["train"])
        training_cosine = F.cosine_similarity(
            features[training_pairs[0]], features[training_pairs[1]]
        )
        raw_cosine_stats = (
            training_cosine.mean().item(),
            training_cosine.std(unbiased=False).clamp_min(1e-6).item(),
        )
    model = ExperimentPairGraphSAGE(
        features.shape[1],
        config.hidden,
        config.dropout,
        raw_cosine_stats,
        config.num_layers,
        config.aggregator,
        config.gru_order,
    ).to(device)  # CPU에서 시드로 초기화한 뒤 옮기므로 초기 가중치는 장치와 무관하게 같음
    features = features.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config.lr, weight_decay=config.weight_decay
    )
    # 이웃 셔플 전용 난수 (dropout 난수열과 분리)
    generator = torch.Generator().manual_seed(config.seed)
    val_pairs, val_labels, val_classes = (
        t.to(device) for t in pair_tensors(splits["val"])
    )
    val_edges, val_weights = (t.to(device) for t in context_edges(splits["train"]))
    rng = random.Random(config.seed)
    best_objective, best_epoch, stale = math.inf, 0, 0
    best_state = None
    history = []
    epoch = 0
    # 폴드 교대 제외로 모든 학습 쌍을 예측 대상에 포함, 현재 폴드의 SIMILAR 연결은 입력 그래프에서 제외
    for epoch in range(1, config.epochs + 1):
        order = list(range(len(splits["train"])))
        rng.shuffle(order)
        fold_indices = [order[i :: config.folds] for i in range(config.folds)]
        train_sse = 0.0
        train_n = 0
        model.train()
        for selected in fold_indices:
            excluded = set(selected)
            context = [
                row
                for index, row in enumerate(splits["train"])
                if index not in excluded
            ]
            targets = [splits["train"][index] for index in selected]
            edges, weights = (t.to(device) for t in context_edges(context))
            pairs, labels, classes = (t.to(device) for t in pair_tensors(targets))
            optimizer.zero_grad()
            predictions = model.score(
                model.encode(features, edges, weights, generator), pairs, features
            )
            rmse = ((predictions - labels).square().mean() + 1e-8).sqrt()
            loss = rmse + config.ordinal_weight * ordinal_loss(
                predictions, classes, config.temperature
            )
            loss.backward()
            optimizer.step()
            train_sse += float((predictions.detach() - labels).square().sum())
            train_n += len(labels)
        model.eval()
        with torch.no_grad():
            val_predictions = model.score(
                model.encode(features, val_edges, val_weights), val_pairs, features
            )
            val_rmse = (val_predictions - val_labels).square().mean().sqrt().item()
            val_ord = ordinal_loss(
                val_predictions, val_classes, config.temperature
            ).item()
            val_objective = val_rmse + config.ordinal_weight * val_ord
            val_exact = (
                ((val_predictions * 5).floor().long().clamp(0, 4) == val_classes)
                .float()
                .mean()
                .item()
            )
        if not math.isfinite(val_objective):
            raise ValueError("non-finite validation objective")
        train_rmse = (train_sse / train_n) ** 0.5
        history.append(
            {
                "epoch": epoch,
                "train_rmse_0_1": train_rmse,
                "val_rmse_0_1": val_rmse,
                "val_ordinal_loss": val_ord,
                "val_objective": val_objective,
                "val_exact_accuracy": val_exact,
            }
        )
        if val_objective < best_objective:
            best_objective, best_epoch, stale = val_objective, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            stale += 1
        if verbose and (epoch == 1 or epoch % 10 == 0):
            print(
                f"epoch={epoch} train_rmse={train_rmse:.4f} "
                f"val_rmse={val_rmse:.4f} val_exact={val_exact:.1%}",
                flush=True,
            )
        if stale >= config.patience:
            break
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        embeddings = model.encode(features, val_edges, val_weights)
        val_predictions = model.score(embeddings, val_pairs, features)
        test_pairs, _, test_classes = (t.to(device) for t in pair_tensors(splits["test"]))
        test_predictions = model.score(embeddings, test_pairs, features)
    return {
        "settings": asdict(config),
        "best_epoch": best_epoch,
        "stopped_epoch": epoch,
        # 조기 종료 없이 epoch 상한까지 간 경우: 학습률이 낮아 덜 수렴했을 수 있음
        "hit_epoch_cap": stale < config.patience,
        "parameters": sum(p.numel() for p in model.parameters()),
        "val": measure(val_predictions, val_classes),
        "test": measure(test_predictions, test_classes),
        "history": history,
    }


# ---------------- 실험 반복 (조건 x 시드) 및 요약 ----------------

SCALAR_METRICS = (
    "rmse_0_1",
    "mae_0_4",
    "exact_accuracy",
    "within_one_accuracy",
    "positive_precision",
    "positive_recall",
    "positive_f1",
    "balanced_accuracy",
)


def _flatten(label: str, report: dict) -> dict:
    row = {
        "condition": label,
        "seed": report["settings"]["seed"],
        "best_epoch": report["best_epoch"],
        "stopped_epoch": report["stopped_epoch"],
        "hit_epoch_cap": report["hit_epoch_cap"],
        "parameters": report["parameters"],
    }
    for part in ("val", "test"):
        for name in SCALAR_METRICS:
            row[f"{part}_{name}"] = report[part][name]
        # 4점 적중 수 (실제 4점을 4점으로 예측한 개수)
        row[f"{part}_hits_4"] = report[part]["confusion_matrix"][4][4]
    return row


def run_grid(
    name: str,
    conditions: dict[str, dict],
    seeds: list[int],
    base: Config = Config(),
    data_path: Path = DATA,
    root: Path = RESULTS,
) -> list[dict]:
    """조건(이름 -> Config에서 바꿀 값) x 시드를 모두 학습하고 결과 행 목록을 반환

    결과는 root/name/<조건>/seed<시드>/metrics.json에 저장하며, 같은 설정으로 이미 학습한
    결과가 있으면 다시 학습하지 않고 읽는다 (중간에 멈춰도 이어서 실행 가능).
    """
    rows = []
    total = len(conditions) * len(seeds)
    for label, overrides in conditions.items():
        for seed in seeds:
            config = replace(base, **overrides, seed=seed)
            out = Path(root) / name / label / f"seed{seed}"
            path = out / "metrics.json"
            report = None
            if path.exists():
                saved = json.loads(path.read_text(encoding="utf-8"))
                if saved["settings"] == asdict(config):
                    report = saved
            if report is None:
                report = train_once(config, data_path)
                history = report.pop("history")
                out.mkdir(parents=True, exist_ok=True)
                path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
                (out / "history.json").write_text(
                    json.dumps(history, indent=2) + "\n", encoding="utf-8"
                )
            rows.append(_flatten(label, report))
            print(
                f"[{len(rows)}/{total}] {label} seed={seed} "
                f"best_epoch={report['best_epoch']} "
                f"val_rmse={report['val']['rmse_0_1']:.4f} "
                f"val_exact={report['val']['exact_accuracy']:.1%}",
                flush=True,
            )
    _write_csv(Path(root) / name / "runs.csv", rows)
    summary = summarize(rows, conditions, base)
    _write_csv(
        Path(root) / name / "summary.csv",
        [{k: v for k, v in row.items() if k != "config"} for row in summary],
    )
    (Path(root) / name / "summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )
    return rows


def summarize(
    rows: list[dict], conditions: dict[str, dict], base: Config = Config()
) -> list[dict]:
    # 조건별 시드 평균·표준편차 (검증 RMSE 평균 오름차순)
    summary = []
    for label, overrides in conditions.items():
        group = [row for row in rows if row["condition"] == label]
        if not group:
            continue
        entry = {
            "condition": label,
            "runs": len(group),
            # 시드·장치는 다음 실험에 넘기지 않음 (다음 실험은 현재 세션의 장치 사용)
            "config": {
                k: v
                for k, v in asdict(replace(base, **overrides)).items()
                if k not in ("seed", "device")
            },
            "device": base.device,
            "epoch_cap_runs": sum(row["hit_epoch_cap"] for row in group),
            "parameters": group[0]["parameters"],
        }
        for key in ["best_epoch"] + [
            f"{part}_{name}"
            for part in ("val", "test")
            for name in (*SCALAR_METRICS, "hits_4")
        ]:
            values = [float(row[key]) for row in group]
            entry[f"{key}_mean"] = statistics.fmean(values)
            entry[f"{key}_std"] = statistics.stdev(values) if len(values) > 1 else 0.0
        summary.append(entry)
    return sorted(summary, key=lambda entry: entry["val_rmse_0_1_mean"])


def best_config(name: str, root: Path = RESULTS) -> tuple[str, dict]:
    """이전 실험 요약에서 검증 RMSE 평균이 가장 낮은 조건의 (이름, 설정) 반환"""
    summary = json.loads((Path(root) / name / "summary.json").read_text(encoding="utf-8"))
    return summary[0]["condition"], summary[0]["config"]


def markdown_table(summary: list[dict], noise: float | None = None) -> str:
    """요약 표 (평균 ± 표준편차). 1위와의 검증 RMSE 차이가 noise보다 작은 조건에 ≈ 표시 (시드 노이즈 수준)"""
    columns = (
        ("val_rmse_0_1", "검증 RMSE", 4),
        ("val_exact_accuracy", "검증 정확", 3),
        ("val_within_one_accuracy", "검증 ±1", 3),
        ("val_positive_f1", "검증 F1(≥3)", 3),
        ("val_hits_4", "검증 4점 적중", 1),
        ("test_rmse_0_1", "테스트 RMSE", 4),
        ("test_exact_accuracy", "테스트 정확", 3),
        ("best_epoch", "best epoch", 1),
    )
    header = "| 순위 | 조건 | " + " | ".join(title for _, title, _ in columns) + " |"
    lines = [header, "|" + "---|" * (len(columns) + 2)]
    best = summary[0]["val_rmse_0_1_mean"] if summary else 0.0
    if noise is None and summary:
        noise = summary[0]["val_rmse_0_1_std"]  # 기본: 1위 조건의 시드 간 표준편차
    for rank, entry in enumerate(summary, 1):
        cells = [
            f"{entry[f'{key}_mean']:.{digits}f} ± {entry[f'{key}_std']:.{digits}f}"
            for key, _, digits in columns
        ]
        mark = ""
        if noise is not None and rank > 1 and entry["val_rmse_0_1_mean"] - best < noise:
            mark = " ≈"
        if entry["epoch_cap_runs"]:
            mark += f" (상한 도달 {entry['epoch_cap_runs']}회)"
        lines.append(f"| {rank} | {entry['condition']}{mark} | " + " | ".join(cells) + " |")
    return "\n".join(lines)


def _write_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def neighborhood_reach(data_path: Path = DATA, max_hops: int = 3) -> list[dict]:
    """학습 그래프(학습 쌍 중 1점 이상)에서 k홉 안에 닿는 이웃 수 통계 (레이어 수 해석용)"""
    artifact = read_artifact(Path(data_path))
    count = len(artifact["recipe_ids"])
    edges, _ = context_edges(artifact["splits"]["train"])
    adjacency = torch.zeros(count, count, dtype=torch.bool)
    adjacency[edges[0], edges[1]] = True
    reach = torch.eye(count, dtype=torch.bool)
    previous = reach.sum(1)
    connected = adjacency.any(1)
    rows = []
    for hops in range(1, max_hops + 1):
        reach = reach | (reach.float() @ adjacency.float()).bool()
        sizes = reach.sum(1) - 1  # 자기 자신 제외
        rows.append(
            {
                "hops": hops,
                "nodes_with_neighbors": int((sizes > 0).sum()),
                "mean_reach_connected": sizes[connected].float().mean().item(),
                "max_reach": int(sizes.max()),
                # 이 홉에서 새 이웃이 생긴 노드 수 (레이어를 하나 더 쌓아 새 정보가 들어오는 노드)
                "nodes_gaining": int((reach.sum(1) > previous).sum()) if hops > 1 else int(connected.sum()),
            }
        )
        previous = reach.sum(1)
    return rows


def plot_summary(summary: list[dict], title: str) -> None:
    # 조건별 검증/테스트 RMSE 평균 ± 표준편차 (Colab 기본 폰트에 한글이 없어 그래프 글자는 영어)
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib이 없어 그래프를 건너뜀")
        return
    labels = [entry["condition"] for entry in summary]
    positions = range(len(labels))
    fig, ax = plt.subplots(figsize=(max(6, 1.2 * len(labels)), 4))
    for offset, part in ((-0.18, "val"), (0.18, "test")):
        ax.errorbar(
            [x + offset for x in positions],
            [entry[f"{part}_rmse_0_1_mean"] for entry in summary],
            yerr=[entry[f"{part}_rmse_0_1_std"] for entry in summary],
            fmt="o", capsize=4, label=part,
        )
    ax.set_xticks(list(positions), labels, rotation=20)
    ax.set_ylabel("RMSE (0-1, lower is better)")
    ax.set_title(title)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_histories(name: str, labels: list[str], seed: int, root: Path = RESULTS) -> None:
    # 조건별 epoch에 따른 검증 RMSE (시드 하나)
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib이 없어 그래프를 건너뜀")
        return
    fig, ax = plt.subplots(figsize=(8, 4))
    for label in labels:
        path = Path(root) / name / label / f"seed{seed}" / "history.json"
        if not path.exists():
            continue
        history = json.loads(path.read_text(encoding="utf-8"))
        ax.plot([h["epoch"] for h in history], [h["val_rmse_0_1"] for h in history], label=label)
    ax.set_xlabel("epoch")
    ax.set_ylabel("val RMSE")
    ax.set_title(f"{name}: learning curves (seed {seed})")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# ======================================================================
# [2] 공통 설정
# ======================================================================
from IPython.display import Markdown, display

SEEDS = [42, 43, 44, 45, 46]  # 조건마다 이 시드들로 반복 학습 (줄이면 빨라지지만 노이즈 판단이 어려워짐)
DATA = Path("data/stage1_rebuild_raw_gte.pt")  # 레시피 GTE 임베딩 + train/val/test 분할
RESULTS = Path("results/experiments")

if not DATA.exists():
    raise FileNotFoundError(f"{DATA}가 없습니다. Colab 파일 탭에서 /content/data/에 올려 주세요.")
print("torch", torch.__version__, "| 장치", DEVICE, "| 시드", SEEDS)

## 실험 1. 이웃 집계 방식

레이어 2, 학습률 0.003 등 나머지는 현재 설정 그대로 두고 GraphSAGE 계층의 **이웃 집계 부분만** 바꿉니다.

| 조건 | 집계 결과 | 엣지 가중치 사용 |
|---|---|---|
| `weighted_mean` | Σ wⱼxⱼ / Σ wⱼ (현재 1단계 방식) | O |
| `mean` | 이웃 벡터의 단순 평균 | X |
| `max` | maxⱼ( wⱼ · ReLU(W xⱼ + b) ) (GraphSAGE pooling) | O |
| `gru_fixed` | 이웃 시퀀스 [xⱼ, wⱼ]를 GRU로 읽은 마지막 은닉 상태, **순서 고정**(엣지 목록 순서, 매번 같음) | O |
| `gru_shuffle` | 위와 같되 **학습 중 forward(폴드 스텝)마다 노드별 이웃 순서를 새로 섞음**. 평가는 고정 순서 | O |

- 엣지 가중치는 학습 쌍의 정답 점수/4(0.25~1.0)입니다. 일반 평균만 가중치를 쓰지 않으므로 `weighted_mean` vs `mean`이 가중치 효과를 보여 줍니다.
- 셔플 난수는 시드로 고정한 별도 생성기를 써서 같은 시드면 결과가 같습니다.
- `gru_weight`(가중치 오름차순으로 고정 정렬, 가장 유사한 이웃을 마지막에 읽음)도 구현돼 있습니다. 비교하려면 아래 주석을 해제하세요.

In [ ]:
# ======================================================================
# 실험 1. 이웃 집계 방식
# ======================================================================
EXPERIMENT_1 = "01_aggregator"
BASE_1 = Config()  # 현재 1단계 설정: 레이어 2, lr 0.003, hidden 64, epoch 상한 100, patience 10
CONDITIONS_1 = {
    "weighted_mean": {"aggregator": "weighted_mean"},
    "mean": {"aggregator": "mean"},
    "max": {"aggregator": "max"},
    "gru_fixed": {"aggregator": "gru", "gru_order": "fixed"},
    "gru_shuffle": {"aggregator": "gru", "gru_order": "shuffle"},
    # "gru_weight": {"aggregator": "gru", "gru_order": "weight"},
}

rows_1 = run_grid(EXPERIMENT_1, CONDITIONS_1, SEEDS, BASE_1, DATA, RESULTS)
summary_1 = summarize(rows_1, CONDITIONS_1, BASE_1)
display(Markdown(markdown_table(summary_1)))
plot_summary(summary_1, "Exp 1. neighbor aggregator")

## 실험 2. GraphSAGE 레이어 수

실험 1에서 검증 RMSE 평균 1위인 집계 방식을 자동으로 가져와 레이어 수만 바꿉니다. 직접 정하려면 `BASE_2`를 바꾸세요(예: `BASE_2 = Config(aggregator="gru", gru_order="shuffle")`).

- 레이어 구성은 기존과 같습니다: `dropout → SAGE → LayerNorm → GELU`를 반복하고, 2번째 레이어부터 residual 연결을 둡니다.
- `LAYERS`에 `0`을 넣으면 그래프를 쓰지 않는 기준 모델(입력 사영 → 쌍 MLP)이 되어 그래프 자체의 기여를 볼 수 있습니다.
- 먼저 나오는 표는 학습 그래프에서 k홉 안에 닿는 이웃 수입니다. 그래프가 희소해서 홉을 늘려도 새 이웃이 생기는 노드가 적으면, 레이어를 더 쌓아도 얻을 정보가 적습니다.

In [ ]:
# ======================================================================
# 실험 2. GraphSAGE 레이어 수
# ======================================================================
EXPERIMENT_2 = "02_num_layers"
best_label_1, best_1 = best_config(EXPERIMENT_1, RESULTS)  # 실험 1의 1위 설정
BASE_2 = Config(**best_1)
LAYERS = [1, 2, 3]  # 0을 추가하면 그래프 미사용 기준 모델
CONDITIONS_2 = {f"L{n}": {"num_layers": n} for n in LAYERS}
print(f"집계 방식: {BASE_2.aggregator} (gru_order={BASE_2.gru_order}) | 실험 1의 1위: {best_label_1}")

reach = neighborhood_reach(DATA, max_hops=max(max(LAYERS), 1))
display(Markdown(
    "| 홉 | 이웃 있는 노드 | 평균 이웃 수(이웃 있는 노드) | 최대 이웃 수 | 이 홉에서 새 이웃이 생긴 노드 |\n"
    "|---|---|---|---|---|\n"
    + "\n".join(
        f"| {r['hops']} | {r['nodes_with_neighbors']} | {r['mean_reach_connected']:.2f} "
        f"| {r['max_reach']} | {r['nodes_gaining']} |"
        for r in reach
    )
))

rows_2 = run_grid(EXPERIMENT_2, CONDITIONS_2, SEEDS, BASE_2, DATA, RESULTS)
summary_2 = summarize(rows_2, CONDITIONS_2, BASE_2)
display(Markdown(markdown_table(summary_2)))
plot_summary(summary_2, f"Exp 2. number of layers ({BASE_2.aggregator})")

## 실험 3. 학습률 (AdamW)

실험 2의 1위 설정(집계 방식·레이어 수)을 자동으로 가져와 학습률만 바꿉니다. 직접 정하려면 `BASE_3`을 바꾸세요.

- 폴드가 5개라 **epoch당 파라미터 업데이트는 5번**뿐입니다. 학습률이 낮으면 기존 상한(100 epoch) 안에 수렴하지 못하므로 이 실험은 상한을 `EPOCHS`(기본 300)로 늘립니다. 상한은 학습 경로를 바꾸지 않으므로 100 epoch 전에 조기 종료되는 실행은 결과가 같습니다(설정 값이 달라 한 번 더 학습할 뿐입니다). 낮은 학습률에서는 epoch당 개선 폭이 노이즈보다 작아 patience 10에 일찍 멈출 수도 있으니 학습 곡선의 끝부분도 확인하세요.
- 조기 종료 없이 상한까지 간 실행은 표에 "(상한 도달 n회)"로 표시합니다. 이런 조건은 덜 수렴했을 수 있으니 `EPOCHS`를 늘려 다시 보세요.
- weight decay(0.001)는 고정합니다. 마지막 그래프는 학습률별 검증 RMSE 학습 곡선(첫 번째 시드)입니다.

In [ ]:
# ======================================================================
# 실험 3. 학습률
# ======================================================================
EXPERIMENT_3 = "03_learning_rate"
best_label_2, best_2 = best_config(EXPERIMENT_2, RESULTS)  # 실험 2의 1위 설정
EPOCHS = 300  # epoch 상한 (patience는 그대로 10)
BASE_3 = replace(Config(**best_2), epochs=EPOCHS)
LRS = [3e-4, 1e-3, 3e-3, 1e-2, 3e-2]
CONDITIONS_3 = {f"lr={lr:g}": {"lr": lr} for lr in LRS}
print(
    f"집계 방식: {BASE_3.aggregator} (gru_order={BASE_3.gru_order}) | "
    f"레이어: {BASE_3.num_layers} | 실험 2의 1위: {best_label_2}"
)

rows_3 = run_grid(EXPERIMENT_3, CONDITIONS_3, SEEDS, BASE_3, DATA, RESULTS)
summary_3 = summarize(rows_3, CONDITIONS_3, BASE_3)
display(Markdown(markdown_table(summary_3)))
plot_summary(summary_3, "Exp 3. learning rate")
plot_histories(EXPERIMENT_3, list(CONDITIONS_3), seed=SEEDS[0], root=RESULTS)

## 결과 다운로드

`results/experiments/` 전체를 zip으로 묶어 내려받습니다(Colab 전용).

In [ ]:
import shutil

archive = shutil.make_archive("stage1_graph_experiments", "zip", RESULTS)
try:
    from google.colab import files

    files.download(archive)
except ImportError:
    print(f"Colab이 아니므로 다운로드 대신 파일만 만듦: {archive}")